In [ ]:
import sys
sys.path.append('../src')
from circuit_postprocess import *
from should_be_stdlib import *
from neurodata import *
from circuits import *
from data import *

In [ ]:
from tqdm.notebook import tqdm
from matplotlib import pyplot as plt
from matplotlib import gridspec
import numpy as np
import seaborn as sns

# data loading

In [ ]:
qf_matrix = {
    k: pd.read_csv(datapath(f'results_simulator_{k}.csv'), index_col=0)
    for k in [
        'ang',
        'ang-qft',
        'amp',
        'amp-qft'
    ]
}

In [ ]:
corrs_df = pd.read_csv(datapath('results_correlation-pearson.csv'), index_col=0)
euclidean_distance_df = pd.read_csv(datapath('results_euclidean.csv'), index_col=0)
euclidean_distance_ifft_df = pd.read_csv(datapath('results_euclidean-ifft.csv'), index_col=0)
classical_fidelity_df = pd.read_csv(datapath('results_classical-fidelity.csv'), index_col=0)

# Cross-correlation btrw multiple matrices

In [ ]:
# Create sample matrices
matrices = [
    mirror_matrix(m.fillna(0).to_numpy())
    for m in [
        corrs_df,
        euclidean_distance_df,
        classical_fidelity_df,
        # euclidean_distance_ifft_df, # basically the same
        qf_matrix['ang'],
        # qpu_ang,
        # qf_matrix['ang-qft'], # basically the same
        qf_matrix['amp'],
        # qpu_amp,
        # qpu_amd_ddd,
        qf_matrix['amp-qft'],
        # qpu_amp_qft,
        # qpu_amp_qft_ddd,
    ]
]
N = len(matrices)
titles = [
    'Correlation',
    'Fidelity',
    'Euclidean',
    'Angle',
    # 'Angle+IBM',
    'Amp',
    # 'Amp+IBM',
    # 'Amp+IBM+DDD',
    'AmpQFT',
    # 'AmpQFT+IBM',
    # 'AmpQFT+IBM+DDD',
]

In [ ]:
fig = plt.figure(figsize=(N * 2, N * 2))
gs = gridspec.GridSpec(N, N, wspace=0.2, hspace=0.2)

# def replace_bottom_triangle_with_nan(matrix):
#     new_matrix = matrix.copy()
#     n = matrix.shape[0]
#     mask = np.tril(np.ones((n, n), dtype=bool))
#     new_matrix[mask] = None
#     return new_matrix

# Plot original heatmaps on the main diagonal and correlation heatmaps
for i in range(N):
    for j in range(i, N):
        if i == j:
            # Plot original heatmaps on the main diagonal
            ax = fig.add_subplot(gs[i, j])
            upper_triangle_matrix = matrices[i]
            # upper_triangle_matrix = replace_bottom_triangle_with_nan(upper_triangle_matrix)  # to eliminate lower triangle
            sns.heatmap(upper_triangle_matrix, ax=ax, cbar=False, square=True, cmap='magma')  # , vmin=0, vmax=1)

        elif i < j:
            # Plot correlation heatmaps in the upper triangle
            ax = fig.add_subplot(gs[i, j])
            corr = np.corrcoef(matrices[i], matrices[j])
            sns.heatmap(corr[len(matrices[i]):, :len(matrices[j])], ax=ax, cbar=False, square=True, cmap='magma')  # , vmin=0, vmax=1)

        # clear labels
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_xlabel('')
        ax.set_ylabel('')

        if i == 0:
            ax.set_title(titles[j], fontsize=12)  # Set title for top row
        if i == j:
            ax.set_ylabel(titles[i], fontsize=12, rotation=0, ha='right')  # Set ylabel for left column

fig.suptitle('Cross-correlation', fontsize=24)
fig.subplots_adjust(top=0.93)  # move suptitle down
plt.tight_layout()

plt.savefig(figspath('cross-correlation.png'), dpi=300)
# plt.show() # big figure, dont show it
plt.clf()

## heatmaps of main diagonal specifically

Just the main diagonal into 4 different plots for closer viewing


In [ ]:
plt.rcParams['font.size'] = 16
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes = axes.flatten()
for i in range(len(axes)):
    ax = axes[i]
    sns.heatmap(matrices[i], ax=ax, cmap='magma', square=True)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.set_title(titles[i])
plt.suptitle('Distance matrices', fontsize=20)
plt.savefig(figspath('correlation-classical.png'), dpi=300)
plt.show()

In [ ]:
plt.rcParams['font.size'] = 16
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes = axes.flatten()
for j in range(len(axes)):
    ax = axes[j]
    i = j + 3
    sns.heatmap(matrices[i], ax=ax, cmap='magma', square=True)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.set_title(titles[i])
plt.suptitle('Quantum fidelities', fontsize=20)
plt.savefig(figspath('correlation-ang-amp-ampqft.png'), dpi=300)
plt.show()

# statistical testing
- the proper test for 2 distance matrices is actually Mantel's test.
- we can use this if we have *distance* matrices to measure the correlation (-1 to +1)
- https://fukamilab.github.io/BIO202/06-C-matrix-comparison.html
- says spearman method should be use when non-linearity is expected, which we use here
- as implemented in skbio, mantel's test requires the diagonals to be 0
- this means that we should invert state fidelity (1-fidelity) bc the middle diagonal is 1, not 0
- The diagonal for state fidelity is not exactly 0, but it's very close to 0 so we can use Mantel's test anyways

In [ ]:
from skbio.stats.distance import mantel, pwmantel

# wrapping to fill diagonals to 0
def mantel_test(a, b):
    # a and b should be PD dataframes
    a_np = a.to_numpy() if type(a) is pd.DataFrame else a
    b_np = b.to_numpy() if type(b) is pd.DataFrame else b
    np.fill_diagonal(a_np, 0)  # in-place ops
    np.fill_diagonal(b_np, 0)
    return mantel(a_np, b_np, method='spearman', seed=0)


def pw_mantel_test(dfs, labels=None):
    # a and b should be PD dataframes
    dfs_np = [d.to_numpy() if type(d) is pd.DataFrame else d.copy() for d in dfs]
    for d in dfs_np:
        np.fill_diagonal(d, 0)
    return pwmantel(dfs_np, method='spearman', labels=labels, seed=0)

In [ ]:
stat_result = pw_mantel_test(
    matrices,
    labels=titles,
)
# rank correlation shows how correlated the ranks of data are
# Euclidean is almost identical to EiFFT
stat_result_table = stat_result[['statistic', 'p-value']]
stat_result_table

In [ ]:
statistic_matrix = stat_result_table.reset_index().pivot_table(index='dm1', columns='dm2')['statistic']
# sort rows by number of NaNs
statistic_matrix = statistic_matrix.loc[statistic_matrix.isnull().sum(axis=1).sort_values(ascending=False).index]
# then sort columns
statistic_matrix = statistic_matrix[statistic_matrix.isnull().sum(axis=0).sort_values(ascending=False).index]
statistic_matrix.to_csv(datapath('mantel-test.csv'))

In [ ]:
plt.rcParams['font.size'] = 16
fig, ax = plt.subplots(figsize=[x*2 for x in statistic_matrix.shape])
sns.heatmap(np.abs(statistic_matrix), annot=True, cmap='magma_r', fmt='.3f')
plt.title('Mantel test statistic (absolute value)', fontsize=20)
plt.xlabel(None)
plt.ylabel(None)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(figspath('mantel-test.png'), dpi=300)
plt.show()

In [ ]:
pvalue_matrix = stat_result_table.reset_index().pivot_table(index='dm1', columns='dm2')['p-value']
# sort rows by number of NaNs
pvalue_matrix = pvalue_matrix.loc[pvalue_matrix.isnull().sum(axis=1).sort_values(ascending=False).index]
# then sort columns
pvalue_matrix = pvalue_matrix[pvalue_matrix.isnull().sum(axis=0).sort_values(ascending=False).index]
pvalue_matrix.to_csv(datapath('mantel-test-pvalue.csv'))

In [ ]:
plt.rcParams['font.size'] = 16
fig, ax = plt.subplots(figsize=[x*2 for x in pvalue_matrix.shape])

sns.heatmap(pvalue_matrix, annot=True, cmap='magma_r', fmt='.3f')
plt.plot(size=(12, 10))
plt.title('Mantel test (p-value)', fontsize=20)
plt.xlabel(None)
plt.ylabel(None)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(figspath('mantel-test-pvalue.png'), dpi=300)
plt.show()